# Problem Statement 1

## Objective : API Data Retrieval and Storage

The objective of this project is to  You are tasked with fetching data from an external REST API, storing it in a local SQLite database, and displaying the retrieved data. The API provides a list of books in JSON format with attributes like title, author, and publication year.

### Import Required Libraries

In [1]:
import requests
import sqlite3
import pandas as pd

###Fetching data from an external REST API

In [2]:
url="https://openlibrary.org/search.json"

In [3]:
req = {
    "q": "biography",
    "limit": 25,
    "fields": "title,author_name,first_publish_year"}

In [4]:
response = requests.get(url, params=req)

In [5]:
response

<Response [200]>

### Converting in json format

In [6]:
data=response.json()

### Getting books data from json

In [7]:
books = data["docs"]

len(books)

25

### A Quick Look at the Book Data

In [8]:
books[:3]

[{'author_name': ['Virginia Woolf'],
  'first_publish_year': 1928,
  'title': 'Orlando'},
 {'author_name': ['Walter Isaacson'],
  'first_publish_year': 2011,
  'title': 'Steve Jobs'},
 {'author_name': ['Virginia Woolf',
   'Eileen Atkins',
   'Katyuli Lloyd',
   'Jose KING',
   'Jordi Fernando',
   'Rafael Vázquez Zamora'],
  'first_publish_year': 1694,
  'title': 'Flush'}]

## Establish SQLite Database Connection

In [9]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

print("Database connected successfully")

Database connected successfully


### Creating table

In [10]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        author TEXT,
        publication_year INTEGER
    )
""")

print("Table created successfully")

Table created successfully


### Storing records

In [11]:
book_records = []

for book in books:
  title = book.get('title',['unknown'])
  author = ", ".join(book.get('author_name',['unknown']))
  year = book.get("first_publish_year")
  book_records.append((title, author, year))

In [12]:
book_records

[('Orlando', 'Virginia Woolf', 1928),
 ('Steve Jobs', 'Walter Isaacson', 2011),
 ('Flush',
  'Virginia Woolf, Eileen Atkins, Katyuli Lloyd, Jose KING, Jordi Fernando, Rafael Vázquez Zamora',
  1694),
 ('Zero', 'Charles Seife', 2000),
 ('Madame Curie', 'Eve Curie', 1937),
 ('Franz Kafka', 'Brod, Max', 1937),
 ('Das Mädchen Anne Frank', 'Melissa Müller', 1998),
 ('Virginia Woolf', 'Quentin Bell', 1972),
 ('Leonardo da Vinci',
  'Walter Isaacson, Jordi Ainaud i Escudero, Beruryah Ben Barukh',
  2017),
 ('Churchill', 'Roy Jenkins', 2001),
 ('COD; A BIOGRAPHY OF THE FISH THAT CHANGED THE WORLD',
  'Mark Kurlansky, S. D. Schindler',
  1997),
 ('Dictionary of National Biography',
  'George Murray Smith, Sir Sidney Lee, Sir Leslie Stephen',
  1885),
 ('Jerusalem', 'Simon Sebag-Montefiore', 2011),
 ('Tolkien', 'Humphrey Carpenter, Pierre Alien', 1977),
 ('Rant', 'Chuck Palahniuk', 1998),
 ('Stalin', 'Robert Service', 2004),
 ('John Adams', 'David McCullough', 2001),
 ('The Beatles', 'Bob Spitz'

### inserting data in books table

In [13]:
cursor.executemany("""
    INSERT INTO books (title, author, publication_year)
    VALUES (?, ?, ?)
""", book_records)

conn.commit()

print("Records inserted successfully")

Records inserted successfully


### Retrieving data

In [14]:
cursor.execute("SELECT * FROM books")

records = cursor.fetchall()

In [15]:
records[:2]

[(1, 'Orlando', 'Virginia Woolf', 1928),
 (2, 'Steve Jobs', 'Walter Isaacson', 2011)]

### Create a DataFrame to Display Retrieved Data

In [16]:
import pandas as pd
df=pd.read_sql_query('select * from books' , conn)

In [17]:
df[['title','author','publication_year']].head()

,title,author,publication_year
0,Orlando,Virginia Woolf,1928
1,Steve Jobs,Walter Isaacson,2011
2,Flush,"Virginia Woolf, Eileen Atkins, Katyuli Lloyd, ...",1694
3,Zero,Charles Seife,2000
4,Madame Curie,Eve Curie,1937
